# CMPhysBench Evaluation Report Template

This notebook generates publication-ready evaluation reports for CondensAI using the CMPhysBench benchmark suite.

## Usage

```python
from cmp_expert.eval import run_benchmark, generate_report

# Run full benchmark
results = run_benchmark(model_name="cm-expert-v1", levels=["ug", "grad", "research"])

# Generate report
generate_report(results, output_file="benchmark_report.pdf")
```

In [ ]:
# Install dependencies (if running in Colab)
# !pip install matplotlib pandas numpy seaborn

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## Load Benchmark Results

In [ ]:
# Example: Load results from JSON
results_file = "../results/benchmark_20260318.json"

if Path(results_file).exists():
    with open(results_file, 'r') as f:
        results = json.load(f)
    print(f"Loaded results from {results_file}")
else:
    print(f"Results file {results_file} not found. Using sample data.")
    # Sample data structure
    results = {
        "model": "cm-expert-v1",
        "date": "2026-03-18",
        "levels": {
            "undergraduate": {
                "accuracy": 0.92,
                "n_questions": 100,
                "correct": 92
            },
            "graduate": {
                "accuracy": 0.78,
                "n_questions": 200,
                "correct": 156
            },
            "research": {
                "accuracy": 0.65,
                "n_questions": 100,
                "correct": 65
            }
        },
        "citation_metrics": {
            "precision": 0.82,
            "recall": 0.71,
            "f1": 0.76
        },
        "calibration": {
            "ece": 0.08,
            "n_bins": 10
        }
    }

## Accuracy by Level

In [ ]:
# Create accuracy dataframe
accuracy_data = []
for level, metrics in results["levels"].items():
    accuracy_data.append({
        "Level": level.title(),
        "Accuracy": metrics["accuracy"],
        "Correct": metrics["correct"],
        "Total": metrics["n_questions"]
    })

df_accuracy = pd.DataFrame(accuracy_data)
df_accuracy

In [ ]:
# Plot accuracy by level
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(df_accuracy["Level"], df_accuracy["Accuracy"], color=['#2ecc71', '#3498db', '#e74c3c'])
ax.set_ylim(0, 1.0)
ax.set_ylabel("Accuracy")
ax.set_title(f"CMPhysBench Accuracy by Level - {results['model']}", fontweight='bold')
ax.axhline(y=0.90, color='#2ecc71', linestyle='--', alpha=0.7, label='Target (90%)')
ax.legend()

# Add value labels on bars
for bar, acc in zip(bars, df_accuracy["Accuracy"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f"{acc:.2f}", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("accuracy_by_level.png", dpi=150, bbox_inches='tight')
plt.show()

## Citation Metrics

In [ ]:
# Citation metrics
citation_metrics = results["citation_metrics"]

fig, ax = plt.subplots(figsize=(8, 6))
metrics_names = ['Precision', 'Recall', 'F1']
metrics_values = [citation_metrics['precision'], citation_metrics['recall'], citation_metrics['f1']]
colors = ['#3498db', '#2ecc71', '#e74c3c']

bars = ax.bar(metrics_names, metrics_values, color=colors)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Score")
ax.set_title("Citation Grounding Performance", fontweight='bold')
ax.axhline(y=0.75, color='#e74c3c', linestyle='--', alpha=0.7, label='Target (0.75)')
ax.legend()

for bar, val in zip(bars, metrics_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f"{val:.2f}", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("citation_metrics.png", dpi=150, bbox_inches='tight')
plt.show()

## Calibration Curve (ECE)

In [ ]:
# Expected Calibration Error
ece = results["calibration"]["ece"]

fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(["ECE"], [ece], color='#9b59b6')
ax.set_ylim(0, 0.5)
ax.set_ylabel("Expected Calibration Error")
ax.set_title(f"Uncertainty Calibration (ECE: {ece:.3f})", fontweight='bold')
ax.axhline(y=0.10, color='#e74c3c', linestyle='--', alpha=0.7, label='Target (< 0.10)')
ax.legend()

plt.tight_layout()
plt.savefig("calibration_ece.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"ECE = {ece:.3f} (Target: < 0.10)")
if ece < 0.10:
    print("✓ Model is well-calibrated!")
else:
    print("✗ Model needs better calibration.")

## Comparison with Baselines

In [ ]:
# Comparison with baseline models
baselines = {
    "GPT-4": {"ug": 0.85, "grad": 0.65, "research": 0.45},
    "Claude-3": {"ug": 0.87, "grad": 0.68, "research": 0.48},
    results["model"]: {
        "ug": results["levels"]["undergraduate"]["accuracy"],
        "grad": results["levels"]["graduate"]["accuracy"],
        "research": results["levels"]["research"]["accuracy"]
    }
}

df_compare = pd.DataFrame(baselines)
df_compare = df_compare.T
df_compare

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
levels = ['undergraduate', 'graduate', 'research']
level_titles = ['Undergraduate', 'Graduate', 'Research']

for idx, (level, title) in enumerate(zip(levels, level_titles)):
    ax = axes[idx]
    data = [baselines[model][level] for model in baselines.keys()]
    colors = ['#95a5a6', '#95a5a6', '#3498db'] if level == 'research' else ['#95a5a6', '#95a5a6', '#2ecc71']
    bars = ax.bar(list(baselines.keys()), data, color=colors)
    ax.set_ylim(0, 1.0)
    ax.set_title(f"{title} Level", fontweight='bold')
    ax.set_ylabel("Accuracy")
    
    for bar, val in zip(bars, data):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f"{val:.2f}", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig("baseline_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

## Summary Statistics

In [ ]:
# Generate summary table
summary_data = {
    "Metric": ["Undergraduate Accuracy", "Graduate Accuracy", "Research Accuracy", 
               "Citation Precision", "Citation Recall", "Citation F1", "ECE"],
    "Value": [
        f"{results['levels']['undergraduate']['accuracy']:.2%}",
        f"{results['levels']['graduate']['accuracy']:.2%}",
        f"{results['levels']['research']['accuracy']:.2%}",
        f"{citation_metrics['precision']:.3f}",
        f"{citation_metrics['recall']:.3f}",
        f"{citation_metrics['f1']:.3f}",
        f"{ece:.3f}"
    ],
    "Target": [">90%", ">75%", ">60%", ">0.75", ">0.75", ">0.75", "<0.10"]
}

df_summary = pd.DataFrame(summary_data)
df_summary

## Export Report

In [ ]:
# Save summary to CSV
df_summary.to_csv("benchmark_summary.csv", index=False)
print("Summary saved to benchmark_summary.csv")

# Save full results to JSON
with open("benchmark_full_results.json", 'w') as f:
    json.dump(results, f, indent=2)
print("Full results saved to benchmark_full_results.json")